In [1]:
import warnings
from pyspark.sql import SparkSession, Window
import  pyspark.sql.functions  as F
from pyspark.sql.types import *
import pyspark.sql.dataframe
from datetime import datetime
from IPython.display import display
import pandas as pd
import sys
import os
import json
#sys.path.append(os.path.abspath(".."))
#from utils.kafka_config import KafkaConfig
import yaml
from pathlib import Path
# Configure pandas to show ful output without truncation
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)     # Show all rows
pd.set_option('display.max_colwidth', None) # Don't truncate columns content
pd.set_option('display.width', None)        # Use full width
import re
print(" Libraries imported successfully")
warnings.filterwarnings("ignore")

 Libraries imported successfully


In [2]:
try: 
    from pyspark import SparkContext
    sc = SparkContext._active_spark_context
    if sc: 
        sc.stop()
        print(" Stoped previous SparkContext")
except:
    pass


# spark.sql.catalog.lake                 org.apache.iceberg.spark.SparkCatalog
# spark.sql.catalog.lake.type            hive
# spark.sql.catalog.lake.uri             thrift://hive-metastore:9083
# spark.sql.catalog.lake.warehouse       s3a://lake/warehouse
# hive.metastore.uris                    thrift://hive-metastore:9083

spark = (
    SparkSession
    .builder
    .appName("Transform data from bornze to silve, Batches")
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .config("spark.sql.shuffle.partitions", 4)
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") 
    .config("spark.sql.catalog.lake",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.session.timeZone", "Asia/Riyadh")
    .config("spark.sql.catalog.lake.type", "hive") 
    .config("spark.sql.catalog.lake.uri", "thrift://hive-metastore:9083") 
.config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.sql.catalog.lake.warehouse", "s3a://lake/warehouse") 
    .config("spark.ssl.enabled", "false")
       .config("spark.hadoop.fs.s3a.endpoint", "http://minio-lb:9000")
        .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")

.config("spark.hadoop.fs.s3a.connection.timeout", "60000")
.config("spark.jars.packages", "org.postgresql:postgresql:42.6.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1,org.apache.iceberg:iceberg-aws-bundle:1.6.1,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")  
        .master("local[*]")
    .getOrCreate()
)
spark

/usr/local/lib/python3.12/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found


:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/spark/.ivy2/cache
The jars for the packages stored in: /home/spark/.ivy2/jars
org.postgresql#postgresql added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b1b78fb8-ddaf-4721-87d3-3f36cb4bcc60;1.0
	confs: [default]
	found org.postgresql#postgresql;42.6.0 in central
	found org.checkerframework#checker-qual;3.31.0 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.3 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.3 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	foun

In [3]:

def print_as_df(df, limit = 5):
    if isinstance(df, pyspark.sql.dataframe.DataFrame):
         display(df.limit(limit).toPandas())
    else:
        print('Unknow types. Just spark dataframe is acceptable')





def get_last_watermark(table_name:str, layer:str, prev_layer:str) -> datetime:
    last_watermark = datetime(2026, 5, 1) # default date
    try: 
        last_watermark = spark.sql(f"""
        SELECT COALESCE(
            (SELECT last_watermark FROM lake.{layer}.watermarks WHERE table_name = '{table_name}'),
            (SELECT MIN(ingestion_ts) FROM lake.{prev_layer}.{table_name})
        ) as last_watermark
    """).collect()[0][0]
        return last_watermark
    except Exception as e : 
        print(e, "\ndefault water_mark",last_watermark)
        raise


def flatten_kafka_payload(df:pyspark.sql.dataframe.DataFrame, table_schema):
     #if isinstance(df, pyspark.sql.dataframe.DataFrame):
     return df.withColumn(
        "after_payload_value", from_json(col("after_payload"),table_schema)
                        ).select(
                                "*",
                                "after_payload_value.*",    
                            ).filter(
                                    (col("operation") != 'd') & (col("after_payload").isNotNull() 
                                                                )).drop("after_payload_value", "after_payload", "before_payload")
     
def fetch_bronze_layer_data(table_path_name, water_mark):
    return spark.sql(f""" SELECT * 
                          FROM {table_path_name}
                          WHERE ingestion_ts >= CAST('{water_mark}' AS TIMESTAMP)""")



def update_watermark_table(table_name, layer_path = "lake.silver"):
    try:
        spark.sql(f"""
                    MERGE INTO {layer_path}.watermarks w
                    USING (SELECT '{table_name}' as table_name, CURRENT_TIMESTAMP as last_watermark, CURRENT_TIMESTAMP as updated_at) src
                    ON w.table_name = src.table_name
                    WHEN MATCHED THEN UPDATE SET w.last_watermark = src.last_watermark, w.updated_at = CURRENT_TIMESTAMP
                    WHEN NOT MATCHED THEN INSERT *
                    """)
        print(f"Updating the watermark table {layer_path}.{table_name} Successed ")
    except Exception as e:
        raise Exception(f"Could not update the water mark for {table_name}\n", e)

def run_func(func, *args, **kwargs):
    start_time = datetime.now()

    func(*args, **kwargs)

    end_time = datetime.now()

    duration = (end_time - start_time).total_seconds()

    print(f"Duration: {duration} sec")

In [4]:
import pyspark
print(pyspark.__version__)

print(spark.version)

3.5.3
3.5.3


In [5]:
print(spark.conf.get("spark.sql.catalog.lake.type"))

hive


In [6]:
print_as_df(spark.sql("SHOW NAMESPACES IN lake"), limit=5)

,namespace
0,bronze
1,default
2,silver


In [7]:
print_as_df(spark.sql("SELECT * FROM lake.bronze.orders"), limit=5)

26/08/10 23:16:23 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


,operation,source_ts_ms,source_lsn,source_txid,source_table,source_snapshot,before_payload,after_payload,kafka_topic,kafka_partition,kafka_offset,kafka_timestamp,ingestion_ts


In [3]:
import requests
resp = requests.get("https://spark-master:8480/json/", verify=False)
apps = resp.json().get("activeapps", [])
apps

/usr/local/lib/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'spark-master'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[{'id': 'app-20260801191504-0000',
  'starttime': 1785611704484,
  'name': 'Thrift JDBC/ODBC Server',
  'cores': 1,
  'user': 'spark',
  'memoryperexecutor': 1024,
  'memoryperslave': 1024,
  'resourcesperexecutor': [],
  'resourcesperslave': [],
  'submitdate': 'Sat Aug 01 19:15:04 UTC 2026',
  'state': 'RUNNING',
  'duration': 2063997}]

In [ ]:
spark.sql("""TRUNCATE TABLE  lake.silver.watermarks
                          """)

In [ ]:
print_as_df(spark.sql(""" SELECT MAX(CAST (order_id AS INT) ) FROM lake.silver.orders
                          --WHERE order_id ='2378418'
                          """), limit =5)

In [45]:
print_as_df(spark.sql(""" SELECT COUNT(*) FROM lake.silver.menu_items
                          """), limit =5)

,count(1)
0,22794


In [8]:
print_as_df(spark.sql(""" SELECT count(*) FROM 
                          lake.silver.restaurants
                          """), limit =5)

,count(1)
0,3000


In [9]:
print(spark.sparkContext.getConf().get("spark.jars"))

file:///home/spark/.ivy2/jars/org.postgresql_postgresql-42.6.0.jar,file:///home/spark/.ivy2/jars/org.apache.spark_spark-sql-kafka-0-10_2.12-3.5.3.jar,file:///home/spark/.ivy2/jars/org.apache.iceberg_iceberg-spark-runtime-3.5_2.12-1.6.1.jar,file:///home/spark/.ivy2/jars/org.apache.iceberg_iceberg-aws-bundle-1.6.1.jar,file:///home/spark/.ivy2/jars/org.apache.hadoop_hadoop-aws-3.3.4.jar,file:///home/spark/.ivy2/jars/com.amazonaws_aws-java-sdk-bundle-1.12.262.jar,file:///home/spark/.ivy2/jars/org.checkerframework_checker-qual-3.31.0.jar,file:///home/spark/.ivy2/jars/org.apache.spark_spark-token-provider-kafka-0-10_2.12-3.5.3.jar,file:///home/spark/.ivy2/jars/org.apache.kafka_kafka-clients-3.4.1.jar,file:///home/spark/.ivy2/jars/com.google.code.findbugs_jsr305-3.0.0.jar,file:///home/spark/.ivy2/jars/org.apache.commons_commons-pool2-2.11.1.jar,file:///home/spark/.ivy2/jars/org.apache.hadoop_hadoop-client-runtime-3.3.4.jar,file:///home/spark/.ivy2/jars/org.lz4_lz4-java-1.8.0.jar,file:///home/

26/07/19 19:39:41 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [7]:
print_as_df(spark.sql(""" SELECT * FROM 
                          lake.bronze.orders  
                          --order by completed_at_ts desc
                          """), limit =5)

,operation,source_ts_ms,source_lsn,source_txid,source_table,source_snapshot,before_payload,after_payload,kafka_topic,kafka_partition,kafka_offset,kafka_timestamp,ingestion_ts
0,r,NaN,1,NaN,orders,orders,None,"{""order_id"":625002,""customer_id"":37008,""restaurant_id"":93,""pickup_zone_id"":32,""dropoff_zone_id"":28,""status"":""placed"",""subtotal"":155.03,""delivery_fee"":13.83,""service_fee"":3.70,""discount"":0.00,""total"":172.56,""placed_at"":""2026-05-13T03:51:45.259Z"",""created_at"":""2026-05-13T03:51:45.259Z"",""updated_at"":""2026-05-13T03:51:45.259Z""}",inital_load_orders,NaN,NaN,NaT,2026-07-10 20:12:45.454153
1,r,NaN,1,NaN,orders,orders,None,"{""order_id"":625003,""customer_id"":10689,""restaurant_id"":593,""pickup_zone_id"":43,""dropoff_zone_id"":43,""status"":""placed"",""subtotal"":25.11,""delivery_fee"":13.87,""service_fee"":2.75,""discount"":0.00,""total"":41.73,""placed_at"":""2026-05-13T03:51:45.303Z"",""created_at"":""2026-05-13T03:51:45.304Z"",""updated_at"":""2026-05-13T03:51:45.304Z""}",inital_load_orders,NaN,NaN,NaT,2026-07-10 20:12:45.454153
2,r,NaN,1,NaN,orders,orders,None,"{""order_id"":625004,""customer_id"":32030,""restaurant_id"":69,""pickup_zone_id"":52,""dropoff_zone_id"":48,""status"":""placed"",""subtotal"":41.01,""delivery_fee"":7.58,""service_fee"":3.75,""discount"":0.00,""total"":52.34,""placed_at"":""2026-05-13T03:51:45.347Z"",""created_at"":""2026-05-13T03:51:45.347Z"",""updated_at"":""2026-05-13T03:51:45.347Z""}",inital_load_orders,NaN,NaN,NaT,2026-07-10 20:12:45.454153
3,r,NaN,1,NaN,orders,orders,None,"{""order_id"":625005,""customer_id"":39603,""restaurant_id"":2313,""pickup_zone_id"":26,""dropoff_zone_id"":35,""status"":""placed"",""subtotal"":235.86,""delivery_fee"":8.40,""service_fee"":5.96,""discount"":0.00,""total"":250.22,""placed_at"":""2026-05-13T03:51:45.391Z"",""created_at"":""2026-05-13T03:51:45.391Z"",""updated_at"":""2026-05-13T03:51:45.391Z""}",inital_load_orders,NaN,NaN,NaT,2026-07-10 20:12:45.454153
4,r,NaN,1,NaN,orders,orders,None,"{""order_id"":625006,""customer_id"":36009,""restaurant_id"":979,""pickup_zone_id"":35,""dropoff_zone_id"":29,""status"":""placed"",""subtotal"":133.35,""delivery_fee"":18.02,""service_fee"":3.47,""discount"":0.00,""total"":154.84,""placed_at"":""2026-05-13T03:51:45.434Z"",""created_at"":""2026-05-13T03:51:45.434Z"",""updated_at"":""2026-05-13T03:51:45.434Z""}",inital_load_orders,NaN,NaN,NaT,2026-07-10 20:12:45.454153


# lake.silver.customers

In [8]:
print_as_df(
    spark.sql("select * from lake.bronze.customers"), limit=1
)

,operation,source_ts_ms,source_lsn,source_txid,source_table,source_snapshot,before_payload,after_payload,kafka_topic,kafka_partition,kafka_offset,kafka_timestamp,ingestion_ts
0,r,NaN,1,NaN,customers,customers,None,"{""customer_id"":1,""email"":""hassan.alsaadi.3844854@example.sa"",""full_name"":""Hassan Al-Saadi"",""phone"":""+966528728463"",""city_id"":2,""default_address"":""Jeddah, Saudi Arabia"",""signup_date"":""2024-10-17"",""is_active"":true,""created_at"":""2026-05-12T20:07:56.477Z"",""updated_at"":""2026-05-12T20:07:56.477Z""}",inital_load_customers,NaN,NaN,NaT,2026-06-09 00:30:38.643567


In [8]:
# {"customer_id":1,"email":"hassan.alsaadi.3844854@example.sa","full_name":"Hassan Al-Saadi","phone":"+966528728463","city_id":2,"default_address":"Jeddah, Saudi Arabia","signup_date":"2024-10-17","is_active":true,"created_at":"2026-05-12T20:07:56.477Z","updated_at":"2026-05-12T20:07:56.477Z"}

In [42]:
spark.sql(""" 
CREATE OR REPLACE TABLE lake.silver.customers (
    customer_id        STRING, 
    email              STRING,
    full_name          STRING,
    prev_phone         STRING,
    phone              STRING,
    prev_city_id       STRING,
    city_id            STRING, 
    default_address    STRING,
    is_active          BOOLEAN,
    signup_ts     TIMESTAMP,
    created_at_ts      TIMESTAMP,
    last_source_update_ts TIMESTAMP,
    last_refresh_ts TIMESTAMP
)
USING iceberg 
TBLPROPERTIES (
    'format-version'                  = '2',
    'write.format.default'            = 'parquet',
    'write.parquet.compression-codec' = 'zstd',
    'write.target-file-size-bytes'    = '134217728'
);
""")
customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("email", StringType(), False),
    StructField("full_name", StringType(), False),
    StructField("prev_phone", StringType(), False),
    StructField("phone", StringType(), False),
    StructField("prev_city_id", StringType(), False),
    StructField("city_id", StringType(), False),
    StructField("default_address", TimestampType(), False),
    StructField("is_active", BooleanType(), False),
    StructField("signup_date", TimestampType(), False),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False)

])

In [43]:
spark.sql("DELETE FROM lake.silver.watermarks WHERE table_name = 'customers'")

DataFrame[]

In [16]:
print_as_df(
    spark.sql("select * from lake.bronze.customers"), limit=1
)

,operation,source_ts_ms,source_lsn,source_txid,source_table,source_snapshot,before_payload,after_payload,kafka_topic,kafka_partition,kafka_offset,kafka_timestamp,ingestion_ts
0,r,NaN,1,NaN,customers,customers,None,"{""customer_id"":3,""email"":""turki.alruwais.7138374@example.sa"",""full_name"":""Turki Al-Ruwais"",""phone"":""+966539587039"",""city_id"":2,""default_address"":""Jeddah, Saudi Arabia"",""signup_date"":""2025-05-28"",""is_active"":true,""created_at"":""2026-05-12T20:07:56.477Z"",""updated_at"":""2026-05-12T20:07:56.477Z""}",inital_load_customers,NaN,NaN,NaT,2026-06-17 03:04:36.579130


In [17]:
# {"customer_id":3,"email":"turki.alruwais.7138374@example.sa","full_name":"Turki Al-Ruwais","phone":"+966539587039","city_id":2,"default_address":"Jeddah, Saudi Arabia","signup_date":"2025-05-28","is_active":true,"created_at":"2026-05-12T20:07:56.477Z","updated_at":"2026-05-12T20:07:56.477Z"}

In [103]:
print_as_df(
    spark.sql("select customer_id, count(*) from lake.silver.customers group by customer_id having  count(*)> 1"), limit=5
)

,customer_id,count(1)


In [44]:
def transforming_customers_to_silver(schema, table_name = 'customers'):

    print(f"Start transforming \ntable name: {table_name}, table type: DIM: SCD3, CDC? No, transforming type: MERGE")
    try:    
        
        last_watermark = get_last_watermark(table_name,
                                            layer='silver',
                                            prev_layer='bronze'
                                           )
       
        df_row_data_bronze_layer = fetch_bronze_layer_data(table_path_name = "lake.bronze."+table_name,
                                                            water_mark=last_watermark
                                                          )
        df_menu_items_flatern = flatten_kafka_payload(df = df_row_data_bronze_layer,
                                                       table_schema = schema )
        # remove the duplicates rows 
        window = Window.partitionBy("customer_id").orderBy(col("source_lsn").desc())        
        df_cdc_after = df_menu_items_flatern.withColumn("rn", row_number().over(window)).filter(col("rn") == 1).drop('rn')

        print("Is there Captured data from kafka?",not df_cdc_after.isEmpty())
        try:
            if  df_cdc_after.isEmpty():
                print("No data in df_canged to Upsert")
                return 
            df_cdc_after.createOrReplaceTempView("customers_updates")
            spark.sql("""
                            MERGE INTO lake.silver.customers cr
                            USING customers_updates cru
                                ON cr.customer_id = cru.customer_id
                            WHEN MATCHED AND (
                                                    cr.email != cru.email OR
                                                    cr.full_name != cru.full_name OR
                                                    cr.phone != cru.phone OR
                                                    cr.city_id != cru.city_id OR
                                                    cr.default_address != cru.default_address OR 
                                                    cr.is_active != cru.is_active
                                                ) THEN
                                UPDATE SET
                                   email = cru.email,
                                   full_name = cru.full_name,
                                   phone = cru.phone,
                                   city_id = cru.city_id,
                                   is_active = cru.is_active,
                                   signup_ts = cru.signup_date,
                                   created_at_ts =  cru.created_at,
                                   last_source_update_ts = cru.updated_at,
                                   prev_city_id = CASE WHEN cr.city_id != cru.city_id 
                                                      THEN cr.city_id ELSE cr.prev_city_id 
                                                  END ,
                                   prev_phone = CASE WHEN cr.phone != cru.phone 
                                                   THEN cr.phone ELSE cr.prev_phone 
                                                END,
                                   last_refresh_ts = CURRENT_TIMESTAMP
                           WHEN NOT MATCHED THEN
                               INSERT (
                                   customer_id,
                                   full_name ,
                                   email ,
                                   prev_phone ,
                                   phone ,
                                   city_id,
                                   prev_city_id ,
                                   is_active,
                                   default_address,
                                   signup_ts ,
                                   created_at_ts ,
                                   last_source_update_ts,
                                   last_refresh_ts 
                                    )
                                    VALUES (
                                        cru.customer_id,
                                        cru.full_name,
                                        cru.email,
                                        cru.prev_phone,
                                        cru.phone ,
                                        cru.city_id,
                                        cru.prev_city_id,
                                        cru.is_active,
                                        cru.default_address,
                                        cru.signup_date,
                                        cru.created_at,
                                        cru.updated_at,
                                        CURRENT_TIMESTAMP
                                    )
                                                            
                """)
            
        except Exception as e : 
                raise Exception(e ,f"could not UPSERT the data for table {table_name.replace('_', ' ')}")
        print(f"MERGE {table_name} data from bronze --> silver: successed")
        print("  total of new inserts",df_cdc_after.count())
        update_watermark_table(table_name = table_name)
        print("  water mark have been updated will be >", last_watermark)
    except Exception as e : 
        raise Exception(e ,f"\n transforming {table_name} from bronze --> silver: failed")

In [45]:
transforming_customers_to_silver(schema = customers_schema)

Start transforming 
table name: customers, table type: DIM: SCD3, CDC? No, transforming type: MERGE
Is there Captured data from kafka? True


,operation,source_ts_ms,source_lsn,source_txid,source_table,source_snapshot,kafka_topic,kafka_partition,kafka_offset,kafka_timestamp,ingestion_ts,customer_id,email,full_name,prev_phone,phone,prev_city_id,city_id,default_address,is_active,signup_date,created_at,updated_at
0,r,NaN,1,NaN,customers,customers,inital_load_customers,NaN,NaN,NaT,2026-06-17 03:04:36.579130,1,hassan.alsaadi.3844854@example.sa,Hassan Al-Saadi,None,+966555555555,None,2,NaT,True,2024-10-17,2026-05-12 20:07:56.477,2026-06-09 18:57:57.230
1,r,NaN,1,NaN,customers,customers,inital_load_customers,NaN,NaN,NaT,2026-06-17 03:04:36.579130,10,yazid.aljuhani.5540561@example.sa,Yazid Al-Juhani,None,+966517507864,None,2,NaT,True,2025-11-15,2026-05-12 20:07:56.477,2026-05-12 20:07:56.477
2,r,NaN,1,NaN,customers,customers,inital_load_customers,NaN,NaN,NaT,2026-06-17 03:04:36.579130,100,mohammed.alsheikh.7658197@example.sa,Mohammed Al-Sheikh,None,+966589699989,None,1,NaT,True,2025-05-12,2026-05-12 20:07:56.477,2026-05-12 20:07:56.477
3,r,NaN,1,NaN,customers,customers,inital_load_customers,NaN,NaN,NaT,2026-06-17 03:04:36.579130,1000,hessa.alzahrani.6202492@example.sa,Hessa Al-Zahrani,None,+966554910261,None,2,NaT,True,2024-08-14,2026-05-12 20:07:56.477,2026-05-12 20:07:56.477
4,r,NaN,1,NaN,customers,customers,inital_load_customers,NaN,NaN,NaT,2026-06-17 03:04:36.579130,10000,bandar.alasiri.7697071@example.sa,Bandar Al-Asiri,None,+966530684922,None,5,NaT,False,2025-04-10,2026-05-12 20:07:56.477,2026-05-12 20:07:56.477


MERGE customers data from bronze --> silver: successed
  total of new inserts 20900
Updating the watermark table lake.silver.customers Successed 
  water mark have been updated will be > 2026-06-17 03:04:36.579130


In [75]:
print_as_df(
    spark.sql("select * from lake.silver.customers"), limit=1
)

,customer_id,email,full_name,prev_phone,phone,prev_city_id,city_id,default_address,is_active,signup_ts,created_at_ts,last_source_update_ts,last_refresh_ts
0,1,hassan.alsaadi.3844854@example.sa,Hassan Al-Saadi,None,+966555555555,None,2,None,True,2024-10-17,2026-05-12 20:07:56.477,2026-06-09 18:57:57.230,2026-06-21 23:55:30.399942


# lake.silver.restaurants

In [120]:
spark.sql("DELETE FROM lake.silver.watermarks WHERE table_name = 'restaurants'")

DataFrame[]

## test 

In [134]:
original = {"restaurant_id":1,"name":"Bayt Al Mandi","cuisine_type":"yemeni","city_id":2,"zone_id":31,"address":"43321 Brittany Bypass, Jeddah, Saudi Arabia","rating_avg":4.55,"is_active":True,"onboarded_at":"2025-04-18T12:08:48.858Z","created_at":"2026-05-12T20:07:56.477Z","updated_at":"2026-06-27 01:30:12.214702"}
updated = {"restaurant_id":1,"name":"Bayt Al Mandi","cuisine_type":"saudi","city_id":2,"zone_id":31,"address":"43321 Brittany Bypass, Jeddah, Saudi Arabia","rating_avg":7.55,"is_active":True,"onboarded_at":"2025-04-18T12:08:48.858Z","created_at":"2026-05-12T20:07:56.477Z","updated_at":str(datetime.now())}

In [111]:
updated

{'restaurant_id': 1,
 'name': 'Bayt Al Mandi',
 'cuisine_type': 'saudii242',
 'city_id': 2,
 'zone_id': 31,
 'address': '43321 Brittany Bypass, Jeddah, Saudi Arabia',
 'rating_avg': 7.55,
 'is_active': True,
 'onboarded_at': '2025-04-18T12:08:48.858Z',
 'created_at': '2026-05-12T20:07:56.477Z',
 'updated_at': '2026-06-27 01:36:01.633339'}

In [135]:
df = spark.sql(""" SELECT * FROM lake.bronze.restaurants""")
print_as_df(df, limit=2)

,operation,source_ts_ms,source_lsn,source_txid,source_table,source_snapshot,before_payload,after_payload,kafka_topic,kafka_partition,kafka_offset,kafka_timestamp,ingestion_ts
0,r,NaN,1,NaN,restaurants,restaurants,None,"{""restaurant_id"":1,""name"":""Bayt Al Mandi"",""cuisine_type"":""yemeni"",""city_id"":2,""zone_id"":31,""address"":""43321 Brittany Bypass, Jeddah, Saudi Arabia"",""rating_avg"":4.55,""is_active"":true,""onboarded_at"":""2025-04-18T12:08:48.858Z"",""created_at"":""2026-05-12T20:07:56.477Z"",""updated_at"":""2026-05-12T20:07:56.477Z""}",inital_load_restaurants,NaN,NaN,NaT,2026-06-17 03:05:03.082170
1,r,NaN,1,NaN,restaurants,restaurants,None,"{""restaurant_id"":2,""name"":""Yokari - Branch 20"",""cuisine_type"":""japanese"",""city_id"":2,""zone_id"":25,""address"":""33890 Jennifer Squares, Jeddah, Saudi Arabia"",""rating_avg"":4.05,""is_active"":true,""onboarded_at"":""2025-07-10T13:42:50.858Z"",""created_at"":""2026-05-12T20:07:56.477Z"",""updated_at"":""2026-05-12T20:07:56.477Z""}",inital_load_restaurants,NaN,NaN,NaT,2026-06-17 03:05:03.082170


In [136]:
df_test = df.filter(col("after_payload").contains(str('"restaurant_id":1,"name":"Bayt Al Mandi","cuisine_type":"yemeni","city_id":2'))).withColumn("after_payload", lit(json.dumps(updated))).withColumn("ingestion_ts", to_timestamp(lit(datetime.now())))
#print_as_df(df_test,limit=1)

In [ ]:
{"restaurant_id":1,"name":"Bayt Al Mandi","cuisine_type":"yemeni","city_id":2,"zone_id":31,"address":"43321 Brittany Bypass, Jeddah, Saudi Arabia","rating_avg":4.55,"is_active":true,"onboarded_at":"2025-04-18T12:08:48.858Z","created_at":"2026-05-12T20:07:56.477Z","updated_at":"2026-05-12T20:07:56.477Z"}

In [121]:
spark.sql(""" 
CREATE OR REPLACE TABLE lake.silver.restaurants (
    restaurant_id        STRING, 
    name                 STRING,
    cuisine_type         STRING,
    city_id              STRING,
    zone_id              STRING,
    address              STRING,
    rating_avg           DECIMAL, 
    is_active            BOOLEAN,
    is_current           BOOLEAN,
    eff_start_ts       TIMESTAMP,
    eff_end_ts         TIMESTAMP,
    onboarded_at         TIMESTAMP,
    created_at_ts        TIMESTAMP,
     last_source_update_ts TIMESTAMP,
    last_refresh_date_ts TIMESTAMP
)
USING iceberg 
TBLPROPERTIES (
    'format-version'                  = '2',
    'write.format.default'            = 'parquet',
    'write.parquet.compression-codec' = 'zstd',
    'write.target-file-size-bytes'    = '134217728'
);
""")
restaurant_schema = StructType([
    StructField("restaurant_id", StringType(), False),
    StructField("name", StringType(), False),
    StructField("cuisine_type", StringType(), False),
    StructField("city_id", StringType(), False),
    StructField("zone_id", StringType(), False),
    StructField("address", StringType(), False),
    StructField("rating_avg", DecimalType(), False),
    StructField("is_active", BooleanType(), False),
    StructField("is_current", BooleanType(), False),
    StructField("eff_start", TimestampType(), False),
    StructField("eff_end", TimestampType(), False),
    StructField("onboarded_at", TimestampType(), False),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False)

])

In [131]:
def transforming_restaurants_to_silver(schema, table_name = 'restaurants'):
    
    print(f"Start transforming \ntable name: {table_name}, table type: DIM: SCD2, CDC? No, transforming type: MERGE")
    try:    
        
        last_watermark = get_last_watermark(table_name,
                                            layer='silver',
                                            prev_layer='bronze'
                                           )       
        df_row_data_bronze_layer = fetch_bronze_layer_data(table_path_name = "lake.bronze."+table_name,
                                                            water_mark=last_watermark
                                                          )
        df_row_data_bronze_layer = df_test
        df_menu_items_flatern = flatten_kafka_payload(df = df_row_data_bronze_layer,
                                                       table_schema = schema )
        
        # remove the duplicates rows 
        window = Window.partitionBy("restaurant_id").orderBy(col("source_lsn").desc())        
        df_cdc = df_menu_items_flatern.withColumn("rn", row_number().over(window)).filter(col("rn") == 1).drop('rn')
        
        df_silver_current = spark.sql("SELECT * FROM lake.silver.restaurants WHERE is_current = true")
        df_changed = df_cdc.join(
            df_silver_current,
            on="restaurant_id",
            how="left"
        ).filter(
            df_silver_current.restaurant_id.isNull() |
            (df_cdc.name != df_silver_current.name) |
            (df_cdc.cuisine_type != df_silver_current.cuisine_type) |
            (df_cdc.rating_avg != df_silver_current.rating_avg) |
            (df_cdc.is_active != df_silver_current.is_active) |
            (df_cdc.city_id != df_silver_current.city_id) |
            (df_cdc.zone_id != df_silver_current.zone_id) |
            (df_cdc.address != df_silver_current.address)
        ).select(df_cdc["*"])#.cache()
        
        df_changed.createOrReplaceTempView("restaurants_merge")
        print("  Is there Caputred data from Kafka?",not df_cdc.isEmpty())
        print("  Is there data in the silver layer?",not df_silver_current.isEmpty())
        print("  Is there data after left join df_cdc with df_silver_current?",not df_changed.isEmpty())
        new_rows = df_changed.count()
        try:
            if  df_changed.isEmpty():
                print("  No data in df_canged to Upsert")
                return
            current_timestamp = datetime.now()
            spark.sql("""
                        MERGE INTO lake.silver.restaurants rest
                        USING restaurants_merge rest_updated
                             on rest.restaurant_id = rest_updated.restaurant_id
                             AND rest.is_current = True
                        WHEN MATCHED THEN UPDATE SET 
                                            rest.is_current = False,
                                            rest.eff_end_ts = rest_updated.updated_at,       
                                            rest.last_source_update_ts = rest_updated.updated_at
                        """)

            spark.sql(f"""
                        INSERT INTO lake.silver.restaurants SELECT 
                                        restaurant_id,
                                        name,
                                        cuisine_type,
                                        city_id,
                                        zone_id,
                                        address,
                                        rating_avg,
                                        is_active,
                                        True,
                                        updated_at,
                                        NULL,
                                        onboarded_at,
                                        created_at as created_at_ts,
                                        updated_at as last_source_update_ts,
                                        CURRENT_TIMESTAMP
                                    FROM restaurants_merge rest_updated              
                        """)
        
        except Exception as e:
            raise Exception(f"Error occred in function merge the {table_name.replace('_', ' ')} rows", e)
        
        print(f"MERGE {table_name} data from bronze --> silver: successed")
        print("  total of new inserts",new_rows)
        update_watermark_table(table_name = table_name)
        print("  water mark have been updated will be >", last_watermark)
    except Exception as e : 
        raise Exception(e ,f"\n transforming {table_name} from bronze --> silver: failed")

In [137]:
transforming_restaurants_to_silver(restaurant_schema)

Start transforming 
table name: restaurants, table type: DIM: SCD2, CDC? No, transforming type: MERGE
  Is there Caputred data from Kafka? True
  Is there data in the silver layer? True
  Is there data after left join df_cdc with df_silver_current? True
MERGE restaurants data from bronze --> silver: successed
  total of new inserts 1
Updating the watermark table lake.silver.restaurants Successed 
  water mark have been updated will be > 2026-06-27 01:42:39.294736


In [138]:
print_as_df(
    spark.sql("select * from lake.silver.restaurants where restaurant_id =1 order by eff_start_ts ASC"), limit=5
)

,restaurant_id,name,cuisine_type,city_id,zone_id,address,rating_avg,is_active,is_current,eff_start_ts,eff_end_ts,onboarded_at,created_at_ts,last_source_update_ts,last_refresh_date_ts
0,1,Bayt Al Mandi,yemeni,2,31,"43321 Brittany Bypass, Jeddah, Saudi Arabia",5,True,False,2026-05-12 20:07:56.477000,2026-06-27 01:42:19.295853,2025-04-18 12:08:48.858,2026-05-12 20:07:56.477,2026-06-27 01:42:19.295853,2026-06-27 01:41:21.095620
1,1,Bayt Al Mandi,saudii242,2,31,"43321 Brittany Bypass, Jeddah, Saudi Arabia",8,True,False,2026-06-27 01:42:19.295853,2026-06-27 01:43:20.307454,2025-04-18 12:08:48.858,2026-05-12 20:07:56.477,2026-06-27 01:43:20.307454,2026-06-27 01:42:36.413652
2,1,Bayt Al Mandi,saudi,2,31,"43321 Brittany Bypass, Jeddah, Saudi Arabia",8,True,True,2026-06-27 01:43:20.307454,NaT,2025-04-18 12:08:48.858,2026-05-12 20:07:56.477,2026-06-27 01:43:20.307454,2026-06-27 01:43:37.673851


# lake.silver.zones

In [171]:
print_as_df(spark.sql(""" select * from lake.bronze.zones"""), limit=2)

,operation,source_ts_ms,source_lsn,source_txid,source_table,source_snapshot,before_payload,after_payload,kafka_topic,kafka_partition,kafka_offset,kafka_timestamp,ingestion_ts
0,r,NaN,1,NaN,zones,zones,None,"{""zone_id"":1,""city_id"":1,""zone_name"":""Al Olaya"",""is_active"":true,""created_at"":""2026-05-12T20:07:56.477Z"",""updated_at"":""2026-05-12T20:07:56.477Z""}",inital_load_zones,NaN,NaN,NaT,2026-06-09 21:46:44.260324
1,r,NaN,1,NaN,zones,zones,None,"{""zone_id"":2,""city_id"":1,""zone_name"":""Al Malaz"",""is_active"":true,""created_at"":""2026-05-12T20:07:56.477Z"",""updated_at"":""2026-05-12T20:07:56.477Z""}",inital_load_zones,NaN,NaN,NaT,2026-06-09 21:46:44.260324


In [53]:
spark.sql("DELETE FROM lake.silver.watermarks WHERE table_name = 'zones'")

DataFrame[]

In [ ]:
{"zone_id":1,"city_id":1,"zone_name":"Al Olaya","is_active":True,"created_at":"2026-05-12T20:07:56.477Z","updated_at":"2026-05-12T20:07:56.477Z"}

In [60]:
spark.sql(""" 
CREATE OR REPLACE TABLE lake.silver.zones (
    zone_id              STRING, 
    city_id              STRING,
    zone_name            STRING,
    is_active            BOOLEAN,
    created_at_ts        TIMESTAMP,
    last_source_update_ts TIMESTAMP,
    last_refresh_ts TIMESTAMP
)
USING iceberg 
TBLPROPERTIES (
    'format-version'                  = '2',
    'write.format.default'            = 'parquet',
    'write.parquet.compression-codec' = 'zstd',
    'write.target-file-size-bytes'    = '134217728'
);
""")
zones_schema = StructType([
    StructField("zone_id", StringType(), False),
    StructField("city_id", StringType(), False),
    StructField("zone_name", StringType(), False),
    StructField("is_active", BooleanType(), False),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False)

])

In [61]:
def transforming_zones_to_silver(schema, table_name = 'zones'):
    
    print(f"Start transforming \ntable name: {table_name}, table type: DIM: SCD0, CDC? No, transforming type: OVERWRITE")
    try:    
        
        last_watermark = get_last_watermark(table_name,
                                            layer='silver',
                                            prev_layer='bronze'
                                           )
       
        df_row_data_bronze_layer = fetch_bronze_layer_data(table_path_name = "lake.bronze."+table_name,
                                                            water_mark=last_watermark
                                                          )
        #df_row_data_bronze_layer = df_test
        df_menu_items_flatern = flatten_kafka_payload(df = df_row_data_bronze_layer,
                                                       table_schema = schema )
        
        # remove the duplicates rows 
        window = Window.partitionBy("zone_id").orderBy(col("source_lsn").desc())        
        df_cdc = df_menu_items_flatern.withColumn("rn", row_number().over(window)).filter(col("rn") == 1).drop('rn')
        
        
        df_cdc.createOrReplaceTempView("zones_updated")
        print("  Is there Caputred data from Kafka?",not df_cdc.isEmpty())

        new_rows = df_cdc.count()
        try:
            if  df_cdc.isEmpty():
                print("  No data in df_canged to OVERWRITE")
                return
            spark.sql("""
                        MERGE INTO lake.silver.zones z
                        USING zones_updated zu
                             on z.zone_id = zu.zone_id
                        WHEN MATCHED THEN UPDATE SET 
                                zone_id = zu.zone_id, 
                                city_id = zu.city_id,
                                zone_name = zu.zone_name,
                                is_active = zu.is_active,
                                created_at_ts = zu.created_at,
                                last_source_update_ts = zu.updated_at,
                                last_refresh_ts =  CURRENT_TIMESTAMP
                        WHEN NOT MATCHED THEN 
                            INSERT (
                                    zone_id,
                                    city_id,
                                    zone_name,
                                    is_active,
                                    created_at_ts,
                                    last_source_update_ts,
                                    last_refresh_ts
                                )
                                VALUES (
                                    zu.zone_id,
                                    zu.city_id,
                                    zu.zone_name,
                                    zu.is_active,
                                    zu.created_at,
                                    zu.updated_at,
                                    CURRENT_TIMESTAMP
                                )
                        """)

        
        except Exception as e:
            raise Exception(f"Error occred in function overwrite the {table_name.replace('_', ' ')} rows", e)
        
        print(f"OVERWRITE {table_name} data from bronze --> silver: successed")
        print("  total of new inserts or updated",new_rows)
        update_watermark_table(table_name = table_name)
        print("  water mark have been updated will be >", last_watermark)
    except Exception as e : 
        raise Exception(e ,f"\n transforming {table_name} from bronze --> silver: failed")

In [62]:
transforming_zones_to_silver(schema=zones_schema)


Start transforming 
table name: zones, table type: DIM: SCD0, CDC? No, transforming type: OVERWRITE
  Is there Caputred data from Kafka? True
OVERWRITE zones data from bronze --> silver: successed
  total of new inserts or updated 61
Updating the watermark table lake.silver.zones Successed 
  water mark have been updated will be > 2026-06-17 03:04:54.408346


In [63]:
print_as_df(spark.sql("select * from lake.silver.zones"))

,zone_id,city_id,zone_name,is_active,created_at_ts,last_source_update_ts,last_refresh_ts
0,1,1,Al Olaya,True,2026-05-12 20:07:56.477,2026-05-12 20:07:56.477,2026-06-22 00:07:48.598585
1,10,1,Al Ghadir,True,2026-05-12 20:07:56.477,2026-05-12 20:07:56.477,2026-06-22 00:07:48.598585
2,11,1,Al Sahafah,True,2026-05-12 20:07:56.477,2026-05-12 20:07:56.477,2026-06-22 00:07:48.598585
3,12,1,Al Izdihar,True,2026-05-12 20:07:56.477,2026-05-12 20:07:56.477,2026-06-22 00:07:48.598585
4,13,1,Al Naseem,True,2026-05-12 20:07:56.477,2026-05-12 20:07:56.477,2026-06-22 00:07:48.598585


In [220]:
spark.sql("DELETE FROM lake.silver.watermarks WHERE table_name = 'zones'")

DataFrame[]

# lake.silver.cities

In [64]:
spark.sql("DELETE FROM lake.silver.watermarks WHERE table_name = 'cities'")

DataFrame[]

In [191]:
print_as_df(spark.sql("select * from lake.bronze.cities"))

,operation,source_ts_ms,source_lsn,source_txid,source_table,source_snapshot,before_payload,after_payload,kafka_topic,kafka_partition,kafka_offset,kafka_timestamp,ingestion_ts
0,r,NaN,1,NaN,cities,cities,None,"{""city_id"":1,""city_name"":""Riyadh"",""country_code"":""SA"",""timezone"":""Asia/Riyadh"",""created_at"":""2026-05-12T20:07:56.477Z"",""updated_at"":""2026-05-12T20:07:56.477Z""}",inital_load_cities,NaN,NaN,NaT,2026-06-09 21:38:43.840248
1,r,NaN,1,NaN,cities,cities,None,"{""city_id"":2,""city_name"":""Jeddah"",""country_code"":""SA"",""timezone"":""Asia/Riyadh"",""created_at"":""2026-05-12T20:07:56.477Z"",""updated_at"":""2026-05-12T20:07:56.477Z""}",inital_load_cities,NaN,NaN,NaT,2026-06-09 21:38:43.840248
2,r,NaN,1,NaN,cities,cities,None,"{""city_id"":3,""city_name"":""Mecca"",""country_code"":""SA"",""timezone"":""Asia/Riyadh"",""created_at"":""2026-05-12T20:07:56.477Z"",""updated_at"":""2026-05-12T20:07:56.477Z""}",inital_load_cities,NaN,NaN,NaT,2026-06-09 21:38:43.840248
3,r,NaN,1,NaN,cities,cities,None,"{""city_id"":4,""city_name"":""Medina"",""country_code"":""SA"",""timezone"":""Asia/Riyadh"",""created_at"":""2026-05-12T20:07:56.477Z"",""updated_at"":""2026-05-12T20:07:56.477Z""}",inital_load_cities,NaN,NaN,NaT,2026-06-09 21:38:43.840248
4,r,NaN,1,NaN,cities,cities,None,"{""city_id"":5,""city_name"":""Dammam"",""country_code"":""SA"",""timezone"":""Asia/Riyadh"",""created_at"":""2026-05-12T20:07:56.477Z"",""updated_at"":""2026-05-12T20:07:56.477Z""}",inital_load_cities,NaN,NaN,NaT,2026-06-09 21:38:43.840248


In [ ]:
{"city_id":1,"city_name":"Riyadh","country_code":"SA","timezone":"Asia/Riyadh","created_at":"2026-05-12T20:07:56.477Z","updated_at":"2026-05-12T20:07:56.477Z"}

In [71]:
spark.sql(""" 
CREATE OR REPLACE TABLE lake.silver.cities (
    city_id              STRING, 
    city_name            STRING,
    country_code         STRING,
    timezone             STRING,
    created_at_ts        TIMESTAMP,
    last_source_update_ts TIMESTAMP,
    last_refresh_ts TIMESTAMP
)
USING iceberg 
TBLPROPERTIES (
    'format-version'                  = '2',
    'write.format.default'            = 'parquet',
    'write.parquet.compression-codec' = 'zstd',
    'write.target-file-size-bytes'    = '134217728'
);
""")
cities_schema = StructType([
    StructField("city_id", StringType(), False),
    StructField("city_name", StringType(), False),
    StructField("country_code", StringType(), False),
    StructField("timezone", StringType(), False),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False)

])

In [73]:
def transforming_cities_to_silver(schema, table_name = 'cities'):
    
    print(f"Start transforming \ntable name: {table_name}, table type: DIM: SCD0, CDC? No, transforming type: OVERWRITE")
    try:    
        
        last_watermark = get_last_watermark(table_name,
                                            layer='silver',
                                            prev_layer='bronze'
                                           )
       
        df_row_data_bronze_layer = fetch_bronze_layer_data(table_path_name = "lake.bronze."+table_name,
                                                            water_mark=last_watermark
                                                          )
        #df_row_data_bronze_layer = df_test
        df_cities_flatern = flatten_kafka_payload(df = df_row_data_bronze_layer,
                                                       table_schema = schema )
        
        # remove the duplicates rows 
        window = Window.partitionBy("city_id").orderBy(col("source_lsn").desc())        
        df_cdc = df_cities_flatern.withColumn("rn", row_number().over(window)).filter(col("rn") == 1).drop('rn')
        
        
        df_cdc.createOrReplaceTempView("cities_updated")
        print("  Is there Caputred data from Kafka?",not df_cdc.isEmpty())

        new_rows = df_cdc.count()
        try:
            if  df_cdc.isEmpty():
                print("  No data in df_canged to OVERWRITE")
                return
    # city_id              STRING, 
    # city_name            STRING,
    # country_code         STRING,
    # timezone             STRING,
    # created_at           TIMESTAMP,
    # updated_at TIMESTAMP
            spark.sql("""
                        MERGE INTO lake.silver.cities c
                        USING cities_updated cu
                             on c.city_id = cu.city_id
                        WHEN MATCHED THEN UPDATE SET 
                                city_name = cu.city_name,
                                country_code = cu.country_code,
                                timezone = cu.timezone,
                                created_at_ts = cu.created_at,
                                last_source_update_ts =  cu.updated_at,
                                last_refresh_ts =  CURRENT_TIMESTAMP
                        WHEN NOT MATCHED THEN 
                            INSERT (
                                city_id,
                                city_name,
                                country_code,
                                timezone,
                                created_at_ts,
                                last_source_update_ts,
                                last_refresh_ts
                            )
                            VALUES (
                                cu.city_id,
                                cu.city_name,
                                cu.country_code,
                                cu.timezone,
                                cu.created_at,
                                cu.updated_at,
                                CURRENT_TIMESTAMP
                            )
                        """)

        
        except Exception as e:
            raise Exception(f"Error occred in function overwrite the {table_name.replace('_', ' ')} rows", e)
        
        print(f"OVERWRITE {table_name} data from bronze --> silver: successed")
        print("  total of new inserts or updated",new_rows)
        update_watermark_table(table_name = table_name)
        print("  water mark have been updated will be >", last_watermark)
    except Exception as e : 
        raise Exception(e ,f"\n transforming {table_name} from bronze --> silver: failed")

In [74]:
transforming_cities_to_silver(schema=cities_schema)

Start transforming 
table name: cities, table type: DIM: SCD0, CDC? No, transforming type: OVERWRITE
  Is there Caputred data from Kafka? True
OVERWRITE cities data from bronze --> silver: successed
  total of new inserts or updated 5
Updating the watermark table lake.silver.cities Successed 
  water mark have been updated will be > 2026-06-17 03:04:46.431960


In [226]:
spark.sql("DELETE FROM lake.silver.watermarks WHERE table_name = 'cities'")

DataFrame[]